# 00 データを見る — 何を予測するのか

> 🔰 **はじめての方へ**: このノートから順に読んでください。プログラミングの経験が少なくても、各セルの前に「何をするか」、後に「出力の見方」を書いています。
> セルは上から順に、`Shift + Enter` で実行します。

## このノートの目的
- 気象庁の**1時間ごとの観測値**がどんな表なのかを知る。
- 「**午後の大雨を当てる**」という問題を、機械学習が扱える形（**標本**・**説明変数**・**正解**）に直す流れを知る。
- **陽性（大雨の日）がどれだけ少ないか**を、自分の目で確かめる。

## キーワード
| 用語 | やさしい説明 |
|---|---|
| **標本（サンプル）** | 機械学習が1回で扱う1行分のデータ。ここでは「ある日の、ある時刻（10・11・12時）」が1標本。 |
| **説明変数（特徴量）** | 予測の手がかりにする値。ここでは気温・湿度・風・午前の日照など。 |
| **正解（ラベル）** | 当てたい答え。ここでは「午後1〜6時に大雨があったか」（あり=1、なし=0）。 |
| **陽性** | 正解が1（大雨あり）の標本。**まれな事象**ほど、陽性が少なく、学習も評価も難しい。 |
| **リーク（情報漏れ）** | 予測するときには**まだ分からないはずの情報**を、うっかり説明変数に入れてしまうこと。 |

In [ ]:
# 準備: 必要なライブラリを読み込み、src フォルダの部品（説明変数を作る dataset.py など）を使えるようにします
import sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path("..").resolve() / "src"))   # ノートブックは notebooks/ にあるので、1つ上の src を探す
plt.rcParams["font.family"] = ["Hiragino Sans", "Arial Unicode MS", "sans-serif"]  # 日本語を表示するための設定
plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.width", 140)
import dataset as ds   # 説明変数と正解を作る部品（src/dataset.py）

## 1. 1時間ごとの観測値を見る

気象庁「過去の気象データ検索」の**時別値**を、`data/hourly.csv` に保存してあります（那覇と名護、2010〜2025年の7〜9月）。
まず中身を見てみましょう。

In [ ]:
# 目的: 時別値の表を読み込んで、最初の数行と列の意味を確かめる
hourly = pd.read_csv("../data/hourly.csv", parse_dates=["time"])
print("行数:", len(hourly), "／ 地点:", hourly["station"].unique())
hourly[hourly.station == "naha"].head(5)

**出力の見方**: 1行が「1地点の1時間」です。主な列は次のとおりです。

| 列 | 意味 | 単位 |
|---|---|---|
| `time` | 時刻。**「1時」の行は0〜1時の1時間の値**（前1時間の値）という約束です。「24時」は翌日の0:00と表示されます。 | |
| `rain` | その1時間の降水量（**1時間降水量**） | mm |
| `temp` / `dew` / `humid` | 気温・露点（空気が冷えて水滴になる温度）・湿度 | ℃ / ℃ / % |
| `pres` | 気圧（現地） | hPa |
| `wind` / `wdir` | 風速と風向（16方位で、風が**吹いてくる**向き） | m/s |
| `sun` / `rad` | 日照時間 / 全天日射量（`rad` は**那覇のみ**。名護は観測なし） | h / MJ/㎡ |

> 📝 降水量が「--」のときは**降水なし**（0mm）として扱っています（`src/fetch_hourly.py`）。

## 2. 「大雨」とは — 1日の雨を見てみる

ある日の、沖縄本島の**11地点の1時間降水量**を重ねて描きます。灰色の帯が、予測の対象にする**午後1〜6時**です。
同じ日でも、地点によって雨の降り方はかなり違います。

In [ ]:
# 目的: 「那覇では降らなかったが、本島のどこかで大雨」の日を1つ選び、全地点の雨を重ねて描く
import region_labels as rl                     # 11地点の1時間降水量をそろえて読む部品
alld = rl.load()                                # 列: station, date, hour, rain
lab = pd.read_csv("../data/daily_labels.csv", index_col=0, parse_dates=True)   # 午後1〜6時の最大1時間降水量（地点／地域）
cand = lab[(lab.naha_region >= 30) & (lab.naha_local < 20)]                    # 地域では30mm/h以上、那覇だけでは20mm/h未満の日
day = cand.index[0]                              # ← 演習: cand.index[3] など別の日にしてみよう
print("選んだ日:", day.date(), " 那覇の最大:", lab.loc[day, "naha_local"], "mm/h ／ 地域の最大:", lab.loc[day, "naha_region"], "mm/h")

g = alld[alld.date == day]
fig, ax = plt.subplots(figsize=(9, 4))
for st, gg in g.groupby("station"):
    ax.plot(gg.hour, gg.rain, marker="o", ms=3, lw=1, label=rl.NAMES[st])
ax.axvspan(14, 18, color="gray", alpha=0.15, label="午後1〜6時（表の14〜18）")
ax.axhline(20, color="red", ls="--", lw=0.8)
ax.set_xlabel("時（h 時の行 ＝ h-1時〜h時の降水量）"); ax.set_ylabel("1時間降水量 (mm)")
ax.legend(ncol=3, fontsize=7); ax.set_title(f"{day.date()} の沖縄本島のアメダス")
plt.show()

**出力の見方**: 赤い破線が **20mm/h** です。同じ日でも、線が破線を超える地点とそうでない地点があります。
ここで **「那覇だけで見ると大雨なしだが、本島のどこかでは大雨」** という日があることが分かります。

### 地点の正解と地域の正解
このプロジェクトでは、正解（ラベル）の作り方を2つ用意しました。

| 正解 | 作り方 | 長所・短所 |
|---|---|---|
| **地点の正解** | 那覇（または名護）**1地点**で、午後1〜6時に基準以上の1時間降水量があったか | 素直。ただし**陽性が少なすぎる** |
| **地域の正解** | 那覇なら**周辺6地点**（南部・中部）、名護なら**8地点**（中部・北部）の**どれか1地点でも**基準以上だったか | 陽性が3〜5倍に増える。ただし、**予測の入力は1地点のまま**で、少し無理がある |

## 3. 陽性はどれだけ少ないか

年ごとの**陽性の日数**（午後1〜6時に大雨があった日）を数えます。基準は 20mm/h と 30mm/h の2通りです。

In [ ]:
# 目的: 年ごとの陽性の日数（地点／地域、20mm/h・30mm/h）を数えて棒グラフにする
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
for ax, th in zip(axes, (20, 30)):
    cnt = pd.DataFrame({"那覇（地点）": (lab.naha_local >= th), "那覇（地域）": (lab.naha_region >= th)}).groupby(lab.index.year).sum()
    cnt.plot.bar(ax=ax, rot=60)
    ax.set_title(f"午後1〜6時に {th}mm/h 以上の日数（各年 92日中）")
    ax.set_xlabel("年")
axes[0].set_ylabel("日数")
plt.tight_layout(); plt.show()
print("全期間（16年×92日 = 1,472日）の合計")
print(pd.DataFrame({th: {"那覇 地点": int((lab.naha_local >= th).sum()), "那覇 地域": int((lab.naha_region >= th).sum()),
                         "名護 地点": int((lab.nago_local >= th).sum()), "名護 地域": int((lab.nago_region >= th).sum())} for th in (20, 30)}))

**出力の見方**
- 30mm/h 以上の日は、**1地点では16年で7〜9日**しかありません。地域にしても20〜43日です。
- 1,472日のうち数十日、つまり**陽性率は数％**です。これが「**クラス不均衡**」です。
- この少なさが、このプロジェクトの**主役の難しさ**です。後のノートで、評価の指標や不確かさの見積もりが重要になる理由はここにあります。

## 4. 機械学習が扱える形にする — 標本・説明変数・正解

予測する時刻は **10時・11時・12時** の3回です。つまり、**1日から3つの標本**ができます。
各標本には、**その時刻までの観測だけ**から作った説明変数をつけます（`src/dataset.py` の `build`）。

In [ ]:
# 目的: 標本の表を作り、1日分（3標本）を見る
x = ds.build("naha")                       # 那覇の標本（全期間）
print("標本数:", len(x), "＝ 1,472日 × 3")
cols = ["date", "t", "temp", "humid", "wind", "rain6", "sun_morning", "y20_region", "y30_region"]
x[x.date == day][cols]

**出力の見方**: 同じ日の3行（`t` = 10, 11, 12）は、**説明変数が少しずつ違い、正解（`y20_region`）は同じ**です。
正解は「午後1〜6時に大雨があったか」で、予測時刻によらないからです。

> ⚠️ この**「同じ日の3標本は、ほぼ同じ答え」**が、後で評価を設計するときの落とし穴になります（→ ノート01）。

### リーク（情報漏れ）がないかの確認
説明変数が**本当にその時刻までの観測だけ**から作られているか、確かめましょう。

In [ ]:
# 目的: 標本（t=12）の説明変数を作るのに使える観測と、使ってはいけない観測を表示する
d = hourly[(hourly.station == "naha")].copy()
d["hour"] = d.time.dt.hour.replace(0, 24)
d["date"] = (d.time - pd.Timedelta(minutes=1)).dt.normalize()
one = d[d.date == day].set_index("hour")[["temp", "humid", "wind", "rain"]]
one["使ってよいか（t=12のとき）"] = ["○ 使える" if h <= 12 else "× 未来なので使わない" for h in one.index]
one.loc[8:18]

**出力の見方**: `t=12`（12時）の標本を作るときは、**12時の行まで**の観測しか使いません。13時以降は**未来**です。
`tests/test_dataset.py` では、午後の観測をわざと壊しても説明変数が変わらないことを自動で確かめています（リークがない証拠）。

## 演習
1. **2.** の `cand.index[0]` を `cand.index[5]` に変えて、別の日の雨を見てください（観察点: 地点によって、雨のピークの時刻がずれているか）。
2. **2.** の `lab.naha_local < 20` を `lab.naha_local < 5` に変えて、「那覇ではほぼ降らなかった日」だけを選んでください（観察点: 該当する日は何日あるか。`len(cand)` で確認）。
3. **3.** の `th` に `(20, 30)` の代わりに `(10, 20)` を入れてください（観察点: 基準を下げると、陽性の日数はどう増えるか）。
4. **4.** の `ds.build("naha")` を `ds.build("nago")` に変えてください（観察点: `rad_morning`（全天日射量）の列がどうなっているか。名護は日射量を観測していません）。